# Capítulo 16 — Computación como laboratorio

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## El primer experimento numérico que sorprendió a todo el mundo.

Reproduce el experimento de Fermi, Pasta, Ulam y Tsingou (1955): una cadena de
osciladores no lineales que, en lugar de termalizar, vuelve casi exactamente a
su estado inicial.

La figura responde: ¿qué pasa cuando el ordenador contradice la intuición de
tres físicos de primera fila?

Ejecutar:  python fig_fpu.py

*(script original: `codigo/fig_fpu.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

N = 32                     # osciladores interiores
ALFA = 0.25                # no linealidad cuadrática
DT = 0.05
PASOS = 1_600_000

# Condición inicial: sólo el primer modo excitado
j = np.arange(1, N + 1)
modos = np.arange(1, N + 1)
x = np.sin(np.pi * j / (N + 1))
v = np.zeros(N)


def fuerzas(x):
    xe = np.concatenate(([0.0], x, [0.0]))
    d = np.diff(xe)                        # x_{i+1} - x_i
    f_lineal = d[1:] - d[:-1]
    f_no_lineal = ALFA * (d[1:]**2 - d[:-1]**2)
    return f_lineal + f_no_lineal


def energia_modos(x, v):
    """Energía en cada modo normal de la cadena lineal."""
    s = np.sqrt(2 / (N + 1)) * np.sin(np.pi * np.outer(modos, j) / (N + 1))
    a, adot = s @ x, s @ v
    w = 2 * np.sin(np.pi * modos / (2 * (N + 1)))
    return 0.5 * (adot**2 + (w * a) ** 2)


guardar_cada = 2000
tiempos, historial = [], []
a = fuerzas(x)
for paso in range(PASOS):
    v += 0.5 * DT * a                      # Verlet de velocidades
    x += DT * v
    a = fuerzas(x)
    v += 0.5 * DT * a
    if paso % guardar_cada == 0:
        historial.append(energia_modos(x, v))
        tiempos.append(paso * DT)

historial = np.array(historial)
tiempos = np.array(tiempos)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.2),
                               gridspec_kw={"width_ratios": [1.5, 1]})

for k, color in zip([1, 2, 3, 4, 5],
                    [C.blue, C.red, C.green, C.ochre, C.purple]):
    ax1.plot(tiempos, historial[:, k - 1], color=color, lw=1.0,
             label=f"modo {k}")
ax1.set_xlabel("tiempo")
ax1.set_ylabel("energía del modo")
ax1.set_title("Cadena no lineal de 32 osciladores: sólo el modo 1 excitado")
ax1.legend(fontsize=8, ncol=5, loc="upper center")

# Recurrencia: cuánta energía vuelve al modo 1
e1 = historial[:, 0]
e_total = historial.sum(axis=1)
frac = e1 / e_total
i_rec = np.argmax(frac[len(frac) // 6:]) + len(frac) // 6
ax1.annotate(f"vuelve el {100*frac[i_rec]:.0f} % al modo inicial",
             xy=(tiempos[i_rec], e1[i_rec]),
             xytext=(tiempos[i_rec] * 0.25, e1[i_rec] * 0.72),
             fontsize=8.6, color=C.ink,
             arrowprops=dict(arrowstyle="->", color=C.ink, lw=1.0))

# --- Lo que se esperaba: equipartición -----------------------------------
ax2.bar(modos - 0.2, historial[-1] / e_total[-1], width=0.4, color=C.blue,
        label="al final de la simulación")
ax2.axhline(1 / N, color=C.red, lw=2.0, ls="--",
            label="equipartición esperada")
ax2.set_xlabel("modo"), ax2.set_ylabel("fracción de energía")
ax2.set_title("Lo que se esperaba, y lo que salió")
ax2.legend(fontsize=8)
ax2.set_xlim(0, 12)

print(f"fracción de energía en los 3 primeros modos al final: "
      f"{historial[-1, :3].sum()/e_total[-1]:.3f}")
print(f"recurrencia máxima al modo 1: {100*frac[i_rec]:.1f} % "
      f"en t = {tiempos[i_rec]:.0f}")
print(f"deriva de la energía total: "
      f"{(e_total[-1]-e_total[0])/e_total[0]:.2e}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Cómo se barre un espacio de parámetros sin desperdiciar CPU?

Rejilla, aleatorio e hipercubo latino en 2D, y la cobertura de las
proyecciones unidimensionales.

La figura responde: ¿por qué una rejilla es una idea peor de lo que parece?

Ejecutar:  python fig_muestreo_parametros.py

*(script original: `codigo/fig_muestreo_parametros.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import qmc

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(16)

N = 25
rejilla = np.stack(np.meshgrid(np.linspace(0.1, 0.9, 5),
                               np.linspace(0.1, 0.9, 5)), -1).reshape(-1, 2)
aleatorio = r.random((N, 2))
lhs = qmc.LatinHypercube(d=2, seed=5).random(N)

fig, axes = plt.subplots(2, 3, figsize=(10.6, 5.4),
                         gridspec_kw={"height_ratios": [2.4, 1], "hspace": 0.35})

for col, (puntos, nombre, color) in enumerate([
        (rejilla, "Rejilla $5\\times5$", C.red),
        (aleatorio, "Aleatorio", C.ochre),
        (lhs, "Hipercubo latino", C.green)]):
    ax = axes[0, col]
    ax.plot(puntos[:, 0], puntos[:, 1], "o", color=color, ms=7)
    for k in np.linspace(0, 1, 26):
        ax.axvline(k, color=C.grey, lw=0.3, alpha=0.5)
    ax.set_xlim(0, 1), ax.set_ylim(0, 1), ax.set_aspect("equal")
    ax.set_xlabel("$p_1$"), ax.set_ylabel("$p_2$")
    ax.set_title(nombre, fontsize=10)
    ax.grid(False)

    ax = axes[1, col]
    ax.hist(puntos[:, 0], bins=25, range=(0, 1), color=color, alpha=0.75)
    ax.set_xlabel("proyección sobre $p_1$")
    ax.set_yticks([0, 1, 5])
    valores_distintos = len(np.unique(np.round(puntos[:, 0], 6)))
    ax.set_title(f"{valores_distintos} valores distintos de $p_1$", fontsize=9)
    print(f"{nombre:22s}: {valores_distintos} valores distintos de p1 "
          f"con {len(puntos)} simulaciones")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
